## 1. Obtención y verificación del dataset

Se utiliza el **Speech Commands Dataset v0.02** (Warden, 2018), un conjunto público de
105 829 grabaciones de 35 palabras distintas, pronunciadas por 2 618 hablantes. Cada clip
dura como máximo 1 segundo y está almacenado en formato WAV mono a 16 kHz.

El archivo comprimido se guarda en una carpeta compartida de Google Drive, de modo que todo
el equipo trabaje sobre la misma copia. En cada sesión se descomprime en el disco local de
Colab (`/content/data`), ya que leer miles de archivos pequeños directamente desde Drive es
considerablemente más lento.

De las 35 palabras disponibles se seleccionan únicamente las **10 autorizadas** por el
enunciado, agrupadas según su función en la aplicación:

| Función | Comandos |
|---|---|
| Confirmación | yes, no |
| Navegación | up, down, left, right |
| Control de dispositivo | on, off |
| Control de acción | stop, go |

La celda verifica la cantidad de clips por clase y la presencia de tres elementos necesarios
en etapas posteriores:

- `validation_list.txt` y `testing_list.txt`: definen la división oficial del dataset.
- `_background_noise_/`: grabaciones de ruido ambiental que se usarán en la aumentación.

**Resultado:** las 10 clases suman 38 546 clips, entre 3 723 (*up*) y 4 044 (*yes*) por
clase. La diferencia entre la clase más y la menos frecuente es de apenas un 8,6 %, por lo
que el dataset se considera **balanceado** y no requiere técnicas de rebalanceo como
ponderación de clases o sobremuestreo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/data
!tar -xzf /content/drive/MyDrive/IA_Proyecto/speech_commands_v0.02.tar.gz -C /content/data

import os
DATA_DIR = '/content/data'
CLASSES = ['yes','no','up','down','left','right','on','off','stop','go']

for c in CLASSES:
    print(f"{c:6s}: {len(os.listdir(os.path.join(DATA_DIR, c)))} clips")

print("\nArchivos clave:")
for f in ['validation_list.txt', 'testing_list.txt', '_background_noise_']:
    print(f"  {f}: {'OK' if os.path.exists(os.path.join(DATA_DIR, f)) else 'FALTA'}")

Mounted at /content/drive
yes   : 4044 clips
no    : 3941 clips
up    : 3723 clips
down  : 3917 clips
left  : 3801 clips
right : 3778 clips
on    : 3845 clips
off   : 3745 clips
stop  : 3872 clips
go    : 3880 clips

Archivos clave:
  validation_list.txt: OK
  testing_list.txt: OK
  _background_noise_: OK


## 2. División en entrenamiento, validación y prueba

El dataset se divide en tres subconjuntos con funciones distintas:

- **Entrenamiento (train):** datos con los que el modelo ajusta sus pesos.
- **Validación (val):** se usa durante el entrenamiento para detectar overfitting o
  underfitting y para seleccionar la mejor configuración de hiperparámetros.
- **Prueba (test):** se reserva hasta el final y se evalúa una sola vez sobre los modelos
  seleccionados, para obtener una estimación no sesgada de su desempeño real.

En lugar de una división aleatoria, se utilizan las **listas oficiales** incluidas en el
dataset (`validation_list.txt` y `testing_list.txt`). Estas listas asignan cada grabación a
un subconjunto según una función hash sobre el identificador del hablante, lo que garantiza
que **todas las grabaciones de una misma persona queden en un único subconjunto**.

Esta decisión es metodológicamente importante: si una misma voz apareciera en entrenamiento
y en prueba, el modelo podría aprender características del hablante en lugar de la palabra,
y las métricas de test resultarían optimistas. Con una división independiente del hablante,
el test mide la capacidad de reconocer comandos de **personas que el modelo nunca escuchó**,
que es exactamente la situación de la aplicación móvil. Además, usar la división oficial
permite comparar los resultados con otros trabajos publicados sobre este dataset.

Todo archivo que no aparece en ninguna de las dos listas se asigna a entrenamiento.

**Resultado:**

| Subconjunto | Clips | Porcentaje |
|---|---|---|
| Entrenamiento | 30 769 | 79,8 % |
| Validación | 3 703 | 9,6 % |
| Prueba | 4 074 | 10,6 % |

La proporción se aproxima a 80/10/10. No es exacta porque la asignación se hace por
hablante y no por clip individual.

In [ ]:
def read_list(name):
    with open(os.path.join(DATA_DIR, name)) as f:
        return set(line.strip() for line in f)

val_set  = read_list('validation_list.txt')
test_set = read_list('testing_list.txt')

splits = {'train': [], 'val': [], 'test': []}
for idx, c in enumerate(CLASSES):
    for fn in sorted(os.listdir(os.path.join(DATA_DIR, c))):
        rel = f"{c}/{fn}"
        if rel in val_set:    splits['val'].append((rel, idx))
        elif rel in test_set: splits['test'].append((rel, idx))
        else:                 splits['train'].append((rel, idx))

for k, v in splits.items():
    print(f"{k:5s}: {len(v)} clips")

train: 30769 clips
val  : 3703 clips
test : 4074 clips


## 3. Caché de audio

Leer más de 38 000 archivos WAV individuales es una operación lenta, limitada por la
velocidad del disco y no por la capacidad de cómputo. Para evitar repetir este proceso en
cada sesión y por cada miembro del equipo, los audios se leen **una única vez** y se guardan
en Google Drive como tensores de PyTorch, uno por subconjunto (`train_wave.pt`,
`val_wave.pt`, `test_wave.pt`).

Durante la carga se aplican los siguientes pasos:

1. **Lectura con `soundfile`:** se usa esta librería en lugar de `torchaudio.load` porque las
   versiones recientes de torchaudio cambiaron su mecanismo de carga y presentan
   incompatibilidades en Colab.
2. **Verificación de la frecuencia de muestreo:** se comprueba que todos los clips estén a
   16 kHz, ya que el preprocesamiento posterior depende de ese valor.
3. **Ajuste de longitud a 1 segundo (16 000 muestras):** algunos clips duran menos de un
   segundo. Se completan con ceros al final (*zero padding*) y, si alguno excediera la
   duración, se recorta. Así todos los audios tienen la misma forma, requisito para
   procesarlos en lotes.
4. **Almacenamiento en `int16`:** es el formato original de los WAV. Guardarlos así ocupa la
   mitad de espacio que `float32`, sin pérdida de información. La conversión a punto
   flotante se hace al momento de usarlos.

Cada archivo guarda, además de las señales (`X`, de forma *N × 16 000*) y las etiquetas
(`y`), los nombres de los archivos originales y la lista de clases, lo que permite rastrear
cualquier muestra hasta su grabación de origen.

A partir de este punto, el `.tar.gz` ya no es necesario para entrenar: basta con montar
Drive y cargar el caché.

In [ ]:
import numpy as np, soundfile as sf, torch
from tqdm.auto import tqdm

SR = 16000
CACHE = '/content/drive/MyDrive/IA_Proyecto/cache'
os.makedirs(CACHE, exist_ok=True)

def load_split(items):
    X = np.zeros((len(items), SR), dtype=np.int16)   # padding con ceros a 1 s
    y = np.zeros(len(items), dtype=np.int64)
    for i, (rel, lab) in enumerate(tqdm(items)):
        w, sr = sf.read(os.path.join(DATA_DIR, rel), dtype='int16')
        assert sr == SR, f"{rel} tiene {sr} Hz"
        w = w[:SR]
        X[i, :len(w)] = w
        y[i] = lab
    return torch.from_numpy(X), torch.from_numpy(y)

for k, items in splits.items():
    X, y = load_split(items)
    torch.save({'X': X, 'y': y, 'files': [r for r, _ in items], 'classes': CLASSES},
               f"{CACHE}/{k}_wave.pt")
    print(k, X.shape)

  0%|          | 0/30769 [00:00<?, ?it/s]

train torch.Size([30769, 16000])


  0%|          | 0/3703 [00:00<?, ?it/s]

val torch.Size([3703, 16000])


  0%|          | 0/4074 [00:00<?, ?it/s]

test torch.Size([4074, 16000])


## 4. Generación del dataset crudo (Log-Mel espectrogramas)

Las redes convolucionales (CNN) reciben como entrada matrices 2D con estructura espacial,
como las imágenes. Una señal de audio, en cambio, es una secuencia 1D de amplitudes en el
tiempo. Para poder usar CNN, cada audio se transforma en un **espectrograma Log-Mel**: una
representación 2D donde un eje es el tiempo, el otro la frecuencia, y el valor de cada
punto es la energía en dB. El resultado se trata como una imagen en escala de grises de un
solo canal.

El dataset **crudo** consiste en estos espectrogramas sin ninguna técnica de aumentación.

### Pasos del preprocesamiento

1. **Normalización de amplitud:** las muestras `int16` se dividen entre 32 768 para llevarlas
   al rango [-1, 1].
2. **Transformada de Fourier de tiempo corto (STFT):** la señal se divide en ventanas
   superpuestas y se calcula el espectro de frecuencias de cada una.
   - Ventana (`n_fft = 480`): 480 muestras = **30 ms**. Es un valor estándar en
     reconocimiento de voz, porque en ese intervalo la señal de voz se puede considerar
     aproximadamente estacionaria.
   - Desplazamiento (`hop_length = 160`): 160 muestras = **10 ms** entre ventanas
     consecutivas. Con 1 segundo de audio se obtienen **101 frames** temporales.
3. **Banco de filtros Mel (`n_mels = 64`):** las frecuencias se agrupan en 64 bandas
   distribuidas según la escala Mel, que imita la percepción del oído humano: mayor
   resolución en frecuencias bajas, donde se concentra la información de la voz, y menor en
   frecuencias altas. Esto también reduce la dimensionalidad respecto al espectro completo.
4. **Conversión a decibeles:** se aplica una escala logarítmica, que comprime el amplio rango
   dinámico de la energía y se asemeja a la percepción humana del volumen. El parámetro
   `top_db = 80` limita el rango a 80 dB por debajo del máximo de **cada clip**, evitando
   valores extremos en los silencios.
5. **Estandarización (z-score):** a cada espectrograma se le resta la media y se divide entre
   la desviación estándar. Estos valores se calculan **únicamente con el conjunto de
   entrenamiento** (media = -30,186 dB, desviación = 20,031 dB) y se aplican igual a
   validación y prueba. Calcularlos con todo el dataset filtraría información de val y test
   hacia el entrenamiento (*data leakage*). La estandarización centra los datos en cero,
   lo que facilita y estabiliza el entrenamiento de la red.

### Resultado

Cada audio queda representado por un tensor de forma **1 × 64 × 101** (canal × bandas Mel ×
frames):

| Subconjunto | Forma del tensor |
|---|---|
| Entrenamiento | 30 769 × 1 × 64 × 101 |
| Validación | 3 703 × 1 × 64 × 101 |
| Prueba | 4 074 × 1 × 64 × 101 |

Los espectrogramas se guardan en `float16` para reducir a la mitad el espacio en Drive.
El cálculo se realiza en la GPU por lotes para acelerar el proceso.

### Configuración para el despliegue

Todos los parámetros del pipeline (frecuencia de muestreo, ventana, desplazamiento, bandas
Mel, `top_db`, media, desviación, orden de las clases y forma de entrada) se guardan en
`preproc_config.json`. La aplicación móvil deberá reproducir **exactamente** este mismo
preprocesamiento antes de pasar el audio al modelo ONNX. De lo contrario, el modelo recibiría
datos con una distribución distinta a la del entrenamiento y sus predicciones perderían
validez.

In [ ]:
import torch, torchaudio, json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
CACHE = '/content/drive/MyDrive/IA_Proyecto/cache'

# Parámetros del pipeline (documentarlos en el informe; la app debe replicarlos)
MEL_CFG = dict(sample_rate=16000, n_fft=480, hop_length=160, n_mels=64)
mel   = torchaudio.transforms.MelSpectrogram(**MEL_CFG).to(device)
to_db = torchaudio.transforms.AmplitudeToDB(top_db=80).to(device)

def waves_to_specs(X, bs=1024):
    out = []
    for i in range(0, len(X), bs):
        w = X[i:i+bs].to(device).float() / 32768.0   # int16 -> [-1, 1]
        s = to_db(mel(w.unsqueeze(1)))                # (B, 1, 64, 101)
        out.append(s.cpu())
    return torch.cat(out)

specs = {}
for k in ['train', 'val', 'test']:
    d = torch.load(f"{CACHE}/{k}_wave.pt")
    specs[k] = (waves_to_specs(d['X']), d['y'])
    print(k, specs[k][0].shape)

# Normalización con estadísticas SOLO de train
mean = specs['train'][0].mean().item()
std  = specs['train'][0].std().item()
print(f"mean={mean:.3f}  std={std:.3f}")

for k, (S, y) in specs.items():
    torch.save({'X': ((S - mean) / std).half(), 'y': y}, f"{CACHE}/{k}_mel_crudo.pt")

with open(f"{CACHE}/preproc_config.json", "w") as f:
    json.dump({**MEL_CFG, 'top_db': 80, 'mean': mean, 'std': std,
               'classes': CLASSES, 'input_shape': [1, 1, 64, 101]}, f, indent=2)

train torch.Size([30769, 1, 64, 101])
val torch.Size([3703, 1, 64, 101])
test torch.Size([4074, 1, 64, 101])
mean=-30.186  std=20.031


# Implementación de Modelos

## Modelo A

In [ ]:
"""LeNet-5 implemented in PyTorch and trained on MNIST.

Architecture (classic LeNet-5, adapted for 28x28 MNIST via padding=2 on conv1,
which is equivalent to the original 32x32 input):

    Input (1x28x28)
    -> Conv(1, 6, 5, padding=2) -> Tanh -> AvgPool(2)    # 6x14x14
    -> Conv(6, 16, 5)           -> Tanh -> AvgPool(2)    # 16x5x5
    -> Flatten (400)
    -> Linear(400, 120) -> Tanh
    -> Linear(120, 84)  -> Tanh
    -> Linear(84, 10)

Usage:
    python lenet5.py --epochs 10 --batch-size 64 --lr 1e-3
"""

import argparse

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


class LeNet5(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, padding=2),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Conv2d(6, 16, kernel_size=5),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),
        )
        with torch.no_grad():
            n_flat = self.features(torch.zeros(1, 1, 64, 101)).numel()
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_flat, 120),
            nn.Tanh(),
            nn.Linear(120, 84),
            nn.Tanh(),
            nn.Linear(84, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        total_loss += F.cross_entropy(outputs, labels).item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


def train_model(epochs=10, batch_size=32, alpha=0.001):
    parser = argparse.ArgumentParser(description="Train LeNet-5 on MNIST")
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--data-dir", type=str, default="./data")
    parser.add_argument("--save-path", type=str, default="lenet5_mnist.pt")
    args = parser.parse_args()

    device = torch.device(
        "cuda" if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available()
        else "cpu"
    )
    print(f"Using device: {device}")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    train_set = datasets.MNIST(args.data_dir, train=True, download=True, transform=transform)
    test_set = datasets.MNIST(args.data_dir, train=False, download=True, transform=transform)
    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=1000)

    model = LeNet5().to(device)
    print(model)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

    for epoch in range(1, args.epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, device)
        te_loss, te_acc = evaluate(model, test_loader, device)
        print(
            f"Epoch {epoch:2d}/{args.epochs} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"test loss {te_loss:.4f} acc {te_acc:.4f}"
        )

    torch.save(model.state_dict(), args.save_path)
    print(f"Saved model to {args.save_path}")


#if __name__ == "__main__":
#    main()

## Modelo B

In [ ]:
# Bloque fundamental de MobileNetV2: Inverted Residual
class InvertedResidual(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, expand_ratio=6):
        super(InvertedResidual, self).__init__()

        self.stride = stride

        # Dimension interna expandida: se multiplica por el factor de expansion
        # Ejemplo: 32 canales entrada * 6 = 192 canales internos
        hidden_dim = in_channels * expand_ratio

        # Solo se usa conexion residual si el stride es 1 y los canales coinciden
        # Si no coinciden las dimensiones, no se pueden sumar entrada y salida
        self.use_residual = (stride == 1 and in_channels == out_channels)

        self.block = nn.Sequential(

            # Paso 1: Expand — aumentar canales con Conv 1x1 + BN + ReLU6
            # ReLU6 es mas estable en dispositivos moviles de baja precision
            nn.Conv2d(in_channels, hidden_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),

            # Paso 2: Depthwise Conv 3x3 — filtra espacialmente canal por canal
            # groups=hidden_dim hace que cada canal se filtre de forma independiente
            # Esto reduce el costo computacional drasticamente vs conv estandar
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3,
                      stride=stride, padding=1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),

            # Paso 3: Project — comprimir canales de vuelta SIN activacion
            # Sin ReLU porque en espacios de baja dimension destruye informacion
            nn.Conv2d(hidden_dim, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        if self.use_residual:
            # Suma la entrada original a la salida (shortcut connection)
            # Mejora el flujo de gradientes durante backpropagation
            return x + self.block(x)
        else:
            return self.block(x)

In [ ]:
# Arquitectura MobileNetV2 adaptada para clasificacion de audio
# Entrada: espectrogramas Mel de forma (batch, 1, 128, 32)
# Salida: 10 clases (comandos de voz)
class MobileNetV2Audio(nn.Module):
    def __init__(self, num_classes=10):
        super(MobileNetV2Audio, self).__init__()

        # Primera capa convolucional estandar
        # Convierte 1 canal (espectrograma) a 32 mapas de caracteristicas
        self.first_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True)
        )

        # Bloques Inverted Residual apilados
        # Cada bloque: (in_channels, out_channels, stride, expand_ratio)
        # stride=2 reduce la dimension espacial a la mitad (downsampling)
        # stride=1 mantiene la dimension espacial
        self.features = nn.Sequential(
            InvertedResidual(32, 16,  stride=1, expand_ratio=1), # sin expansion en el primero
            InvertedResidual(16, 24,  stride=2, expand_ratio=6), # downsampling
            InvertedResidual(24, 24,  stride=1, expand_ratio=6), # mismo tamaño, shortcut activo
            InvertedResidual(24, 32,  stride=2, expand_ratio=6), # downsampling
            InvertedResidual(32, 32,  stride=1, expand_ratio=6), # mismo tamaño, shortcut activo
            InvertedResidual(32, 64,  stride=2, expand_ratio=6), # downsampling
            InvertedResidual(64, 64,  stride=1, expand_ratio=6), # mismo tamaño, shortcut activo
        )

        # Ultima conv 1x1 para enriquecer la representacion antes del clasificador
        self.last_conv = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU6(inplace=True)
        )

        # Global Average Pooling: colapsa el mapa espacial a un solo valor por canal
        # Mas eficiente que Flatten para dispositivos moviles
        self.gap = nn.AdaptiveAvgPool2d(1)

        # Dropout para regularizacion y evitar overfitting
        self.dropout = nn.Dropout(p=0.2)

        # Capa de clasificacion final: 128 -> 10 comandos
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.first_conv(x)     # conv inicial: (batch, 1, 128, 32) -> (batch, 32, 128, 32)
        x = self.features(x)       # bloques inverted residual con downsampling
        x = self.last_conv(x)      # enriquecimiento final
        x = self.gap(x)            # (batch, 128, H, W) -> (batch, 128, 1, 1)
        x = x.view(x.size(0), -1)  # flatten -> (batch, 128)
        x = self.dropout(x)        # regularizacion
        x = self.classifier(x)     # salida -> (batch, 10)
        return x


# Verificacion de que el modelo recibe y produce las dimensiones correctas
torch.manual_seed(42)
model_b = MobileNetV2Audio()
x_test = torch.randn(1, 1, 128, 32)
out = model_b(x_test)
print("Output shape:", out.shape)  # debe ser torch.Size([1, 10])

# Cantidad total de parametros del modelo
total_params = sum(p.numel() for p in model_b.parameters())
print(f"Total parametros: {total_params:,}")